# Research Assist · GPU embedding trên Kaggle

Chạy notebook này nếu bạn muốn **Research Assist trên máy mình gửi yêu cầu embedding
tới GPU Kaggle**. Bật **Internet**, chọn **GPU T4** và chạy trong phiên tương tác.
Thêm Kaggle Secret `RESEARCH_GPU_API_KEY` (ít nhất 32 ký tự) và cấp quyền cho notebook.
Nhập lại cùng khóa trong mục **GPU embedding** của Research Assist.

Mặc định dùng **Qwen3-Embedding-0.6B**, có thể chạy ngay. Để dùng LoRA từ notebook
**01**, gắn output làm Kaggle Input rồi trỏ `CHECKPOINT_ZIP` tới `qwen3-adapter.zip`.
Để dùng E5 đã huấn luyện, chọn `MODEL_FAMILY = "e5"` và cung cấp checkpoint E5.
Adapter **reranker** từ notebook 02 dành cho một tác vụ khác.

Bật `START_GPU_SERVER` và chạy lại từ cell cấu hình để cài thư viện, tải model
và kiểm tra API. Sau đó bật
`START_CLOUDFLARE` nếu bạn có quyền dùng đường hầm trên môi trường chạy.
URL tạm chỉ tồn tại trong phiên notebook. Khi xong, bật `STOP_ENDPOINT`
trong cell cuối để dừng server và đường hầm.

In [ ]:
MODEL_FAMILY = "qwen3"  # "qwen3" hoặc "e5"
BASE_MODEL = "Qwen/Qwen3-Embedding-0.6B"  # Đặt khớp model nếu dùng thư mục Qwen3 đầy đủ.
BASE_REVISION = ""  # Tùy chọn: SHA Hugging Face cho base model.
CHECKPOINT_DIR = ""  # Thư mục Qwen3 đầy đủ, SWIFT adapter, hoặc checkpoint E5.
CHECKPOINT_ZIP = ""  # VD: /kaggle/input/my-training/qwen3-adapter.zip.
E5_TOKENIZER_MODEL = "intfloat/multilingual-e5-base"  # Phải khớp checkpoint E5.
MAX_SEQ_LENGTH = 512
PORT = 8008
START_GPU_SERVER = False
START_CLOUDFLARE = False
API_KEY_SECRET_NAME = "RESEARCH_GPU_API_KEY"

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
WORKING_DIR = Path("/kaggle/temp/research-assist-embedding")
if START_GPU_SERVER:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "sentence-transformers==5.1.2", "transformers==4.57.1",
        "fastapi>=0.115,<1", "uvicorn>=0.30,<1", "httpx>=0.27,<1",
        "huggingface-hub==0.36.0", "peft==0.17.1",
        "bitsandbytes==0.48.2", "accelerate==1.11.0",
    ])

In [ ]:
"""Start and stop an optional Cloudflare Quick Tunnel in an interactive notebook."""

import hashlib
import json
import os
import platform
import queue
import re
import shutil
import subprocess
import tempfile
import threading
import time
import urllib.request
from pathlib import Path


def find_cloudflared(working_dir: Path | str) -> str:
    installed = shutil.which("cloudflared")
    if installed:
        return installed
    machine = platform.machine().lower()
    if platform.system() != "Linux" or machine not in {"x86_64", "amd64", "aarch64", "arm64"}:
        raise RuntimeError("Install cloudflared for your operating system and put it on PATH.")
    arch = "amd64" if machine in {"x86_64", "amd64"} else "arm64"
    name = f"cloudflared-linux-{arch}"
    directory = Path(working_dir)
    directory.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(
        "https://api.github.com/repos/cloudflare/cloudflared/releases/latest",
        headers={"User-Agent": "Research-Assist-Notebook"},
    )
    with urllib.request.urlopen(request, timeout=15) as response:
        release = json.load(response)
    asset = next((item for item in release["assets"] if item["name"] == name), None)
    if not asset or not re.fullmatch(r"sha256:[a-fA-F0-9]{64}", asset.get("digest") or ""):
        raise RuntimeError("No published SHA256 digest for cloudflared. Install it manually.")
    digest = asset["digest"].split(":", 1)[1].lower()
    target = directory / name
    if target.is_file():
        with target.open("rb") as existing:
            if hashlib.file_digest(existing, "sha256").hexdigest() == digest:
                target.chmod(0o700)
                return str(target.resolve())
    deadline = time.monotonic() + 90
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=directory, delete=False) as output:
            temporary = Path(output.name)
            checksum = hashlib.sha256()
            with urllib.request.urlopen(asset["browser_download_url"], timeout=15) as response:
                while block := response.read(1024 * 1024):
                    if time.monotonic() >= deadline:
                        raise TimeoutError("Downloading cloudflared took too long.")
                    checksum.update(block)
                    output.write(block)
        if checksum.hexdigest() != digest:
            raise RuntimeError("cloudflared does not match the official release checksum.")
        temporary.chmod(0o700)
        os.replace(temporary, target)
        return str(target.resolve())
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def stop_process(process: subprocess.Popen) -> None:
    """Reap the process even if it ignores a graceful stop."""
    if process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)


def start_quick_tunnel(
    port: int, working_dir: Path | str, *, timeout: float = 60
) -> tuple[subprocess.Popen, str]:
    """Return the owned process and assigned HTTPS URL, or tear it down on error."""
    if not 1 <= port <= 65535 or timeout <= 0:
        raise ValueError("Invalid tunnel port or timeout.")
    executable = find_cloudflared(working_dir)
    process = subprocess.Popen(
        [executable, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    results: queue.Queue[str | None] = queue.Queue()

    def read_output() -> None:
        stream = process.stdout
        if stream is None:
            results.put(None)
            return
        found = False
        with stream:
            for line in stream:
                match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com\b", line)
                if not found and match:
                    found = True
                    results.put(match.group())
        results.put(None)

    try:
        threading.Thread(target=read_output, daemon=True).start()
        try:
            url = results.get(timeout=timeout)
        except queue.Empty:
            raise TimeoutError("No Cloudflare URL received before the timeout.") from None
        if not url or process.poll() is not None:
            raise RuntimeError("cloudflared exited before the tunnel was ready.")
        return process, url
    except BaseException:
        stop_process(process)
        raise

In [ ]:
"""Inference for the SWIFT Qwen3 adapter recipe; identical query/EOS policy to training."""

import hashlib
import json
from pathlib import Path


def adapter_identity(path, max_seq_length):
    root = Path(path)
    digest = hashlib.sha256(f"swift-qwen3-eos-v1:{max_seq_length}".encode())
    for name in ("research_model.json", "adapter_config.json", "adapter_model.safetensors"):
        digest.update(name.encode())
        with (root / name).open("rb") as stream:
            while block := stream.read(1024 * 1024):
                digest.update(block)
    return "research-qwen3-" + digest.hexdigest()[:32]


def read_adapter_manifest(path):
    root = Path(path)
    manifest = json.loads((root / "research_model.json").read_text(encoding="utf-8"))
    if (
        manifest.get("format") != "swift_qwen3_adapter_v1"
        or manifest.get("model_family") != "qwen3"
        or manifest.get("fine_tuned") is not True
        or manifest.get("optimizer_steps", 0) < 1
        or manifest.get("base_model")
        not in {"Qwen/Qwen3-Embedding-4B", "Qwen/Qwen3-Embedding-0.6B"}
        or len(manifest.get("base_revision", "")) != 40
        or not manifest.get("query_instruction")
        or not (root / "adapter_model.safetensors").is_file()
    ):
        raise ValueError("Invalid trained Qwen3 adapter manifest; base fallback is forbidden.")
    return manifest


def extract_adapter_archive(archive_path, destination):
    import zipfile

    root = Path(destination).resolve()
    if root.exists() and any(root.iterdir()):
        raise ValueError("Choose an empty adapter extraction directory.")
    with zipfile.ZipFile(archive_path) as archive:
        size = 0
        for item in archive.infolist():
            target = (root / item.filename.replace("\\", "/")).resolve()
            if not target.is_relative_to(root) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Unsafe adapter archive entry.")
            size += item.file_size
            if size > 2 * 1024**3:
                raise ValueError("Adapter archive exceeds 2 GiB.")
        archive.extractall(root)
    read_adapter_manifest(root)
    return root


class QwenAdapterEncoder:
    """Load a pinned base plus optional adapter. Use base-only solely for baseline evaluation."""

    def __init__(self, base_path, instruction, max_seq_length=512, adapter_path=None):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        self.max_seq_length, self.instruction = max_seq_length, instruction
        self.tokenizer = AutoTokenizer.from_pretrained(str(base_path), trust_remote_code=False)
        self.tokenizer.padding_side = "right"
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.suffix = self.tokenizer.convert_tokens_to_ids("<|endoftext|>")
        if self.suffix is None or self.suffix == self.tokenizer.unk_token_id:
            raise ValueError("Qwen tokenizer must contain the SWIFT endoftext suffix.")
        model = AutoModelForCausalLM.from_pretrained(
            str(base_path),
            trust_remote_code=False,
            local_files_only=True,
            torch_dtype=torch.float16,
            attn_implementation="sdpa",
            device_map={"": 0},
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            ),
        )
        if adapter_path:
            from peft import PeftModel

            model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)
        self.model = model.eval()

    def get_sentence_embedding_dimension(self):
        return int(self.model.config.hidden_size)

    def encode_query(self, texts, **kwargs):
        return self._encode(
            [f"Instruct: {self.instruction}\nQuery:{text}" for text in texts], **kwargs
        )

    def encode_document(self, texts, **kwargs):
        return self._encode(texts, **kwargs)

    def _encode(self, texts, batch_size=2, **kwargs):
        import numpy as np
        import torch
        import torch.nn.functional as functional

        result = []
        with torch.inference_mode():
            for start in range(0, len(texts), batch_size):
                ids = self.tokenizer(
                    texts[start : start + batch_size],
                    add_special_tokens=False,
                    truncation=True,
                    max_length=self.max_seq_length - 1,
                )["input_ids"]
                batch = self.tokenizer.pad(
                    {"input_ids": [row + [self.suffix] for row in ids]},
                    padding=True,
                    return_tensors="pt",
                ).to(self.model.device)
                base = (
                    self.model.get_base_model()
                    if hasattr(self.model, "peft_config")
                    else self.model
                )
                # Use the transformer with adapters; avoid the vocabulary-sized LM head.
                hidden = base.model(**batch, use_cache=False, return_dict=True).last_hidden_state
                last = batch["attention_mask"].sum(dim=1) - 1
                vectors = hidden[torch.arange(len(last), device=hidden.device), last].float()
                result.append(functional.normalize(vectors, dim=1).cpu().numpy())
        return (
            np.concatenate(result)
            if result
            else np.empty((0, self.get_sentence_embedding_dimension()))
        )


def load_swift_qwen_adapter(path, max_seq_length=512):
    from huggingface_hub import snapshot_download

    manifest = read_adapter_manifest(path)
    base = snapshot_download(manifest["base_model"], revision=manifest["base_revision"])
    encoder = QwenAdapterEncoder(
        base, manifest["query_instruction"], max_seq_length, adapter_path=path
    )
    return encoder, adapter_identity(path, max_seq_length)

In [ ]:
if START_GPU_SERVER:
    """Authenticated, bounded E5 inference; usable on a permitted notebook or GPU host."""

    import asyncio
    import hashlib
    import json
    import secrets
    import threading
    from pathlib import Path
    from typing import Literal

    from fastapi import FastAPI, HTTPException, Request
    from pydantic import BaseModel, ConfigDict, ValidationError


    def checkpoint_identity(path, max_seq_length=512, family="e5"):
        """Identity follows weights/tokenizer/config, never the temporary tunnel URL."""
        root = Path(path)
        files = sorted(
            p
            for p in root.rglob("*")
            if p.is_file()
            and p.suffix
            in {
                ".safetensors",
                ".bin",
                ".json",
                ".model",
                ".txt",
            }
            and "checkpoints" not in p.relative_to(root).parts
        )
        if not any(p.suffix in {".safetensors", ".bin"} for p in files):
            raise ValueError("Checkpoint must contain model weights and tokenizer/config files.")
        if family not in {"e5", "qwen3"}:
            raise ValueError("Unsupported embedding family.")
        digest = hashlib.sha256(f"{family}:query:passage:normalized:v1:{max_seq_length}".encode())
        for file in files:
            digest.update(str(file.relative_to(root)).replace("\\", "/").encode())
            with file.open("rb") as stream:
                while block := stream.read(1024 * 1024):
                    digest.update(block)
        return f"research-{family}-" + digest.hexdigest()[:32]


    class EmbeddingInput(BaseModel):
        model_config = ConfigDict(extra="forbid", strict=True)
        model: str
        input: str | list[str]
        input_type: Literal["query", "passage"]


    def create_embedding_app(
        encoder, api_key, model_id, max_seq_length=512, family="e5", tokenizer_model=""
    ):
        if len(api_key) < 32:
            raise ValueError("Use an API key with at least 32 characters.")
        if family not in {"e5", "qwen3"} or not model_id.startswith(f"research-{family}-"):
            raise ValueError("Model ID must match the embedding family.")
        app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
        lock = threading.Lock()
        active = 0
        dimension = int(encoder.get_sentence_embedding_dimension())

        def authorize(request):
            header = request.headers.get("authorization", "")
            if not secrets.compare_digest(header, "Bearer " + api_key):
                raise HTTPException(401, "Invalid API key.")

        @app.get("/health")
        async def health():
            return {"status": "ok"}

        @app.get("/v1/info")
        async def info(request: Request):
            authorize(request)
            return {
                "model": model_id,
                "family": family,
                "tokenizer_model": tokenizer_model,
                "dimension": dimension,
                "max_seq_length": max_seq_length,
                "normalized": True,
                "protocol": "research-embedding-v1",
                "max_batch_size": 32,
            }

        def encode(texts, input_type):
            with lock:
                if family == "e5":
                    method = encoder.encode
                    inputs = [("query: " if input_type == "query" else "passage: ") + t for t in texts]
                else:
                    method = encoder.encode_query if input_type == "query" else encoder.encode_document
                    inputs = texts
                vectors = method(
                    inputs,
                    batch_size=8 if family == "e5" else 2,
                    normalize_embeddings=True,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                )
                return vectors.tolist()

        @app.post("/v1/embeddings")
        async def embeddings(request: Request):
            nonlocal active
            authorize(request)
            if active >= 4:
                raise HTTPException(429, "GPU queue is full; retry later.")
            body = bytearray()
            async for part in request.stream():
                body.extend(part)
                if len(body) > 300000:
                    raise HTTPException(413, "Request is too large.")
            try:
                value = EmbeddingInput.model_validate_json(body)
            except (ValidationError, ValueError):
                raise HTTPException(422, "Invalid embedding request.") from None
            if value.model != model_id:
                raise HTTPException(409, "Checkpoint changed; configure the new model ID and reindex.")
            texts = [value.input] if isinstance(value.input, str) else value.input
            if not 1 <= len(texts) <= 32 or any(
                not text.strip() or len(text) > 16000 for text in texts
            ):
                raise HTTPException(422, "Use 1-32 nonempty texts, at most 16000 characters each.")
            if active >= 4:
                raise HTTPException(429, "GPU queue is full; retry later.")
            # Retain the queue slot and physical lock until the thread actually exits.
            active += 1
            job = asyncio.create_task(asyncio.to_thread(encode, texts, value.input_type))

            def finished(task):
                nonlocal active
                active -= 1
                if not task.cancelled():
                    task.exception()  # Retrieve a late exception after an HTTP cancellation.

            job.add_done_callback(finished)
            try:
                vectors = await asyncio.shield(job)
            except asyncio.CancelledError:
                raise
            except Exception:
                raise HTTPException(
                    503, "GPU inference failed; inspect the private server log."
                ) from None
            return {
                "object": "list",
                "model": model_id,
                "data": [
                    {"object": "embedding", "index": i, "embedding": vector}
                    for i, vector in enumerate(vectors)
                ],
            }

        return app


    def extract_checkpoint(archive, output):
        """Extract a downloaded training checkpoint; reject traversal and symlinks."""
        import zipfile

        root = Path(output).resolve()
        if root.exists() and any(root.iterdir()):
            raise ValueError("Choose an empty checkpoint extraction directory.")
        with zipfile.ZipFile(archive) as zipped:
            total = 0
            for item in zipped.infolist():
                target = (root / item.filename.replace("\\", "/")).resolve()
                if not target.is_relative_to(root) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Unsafe checkpoint archive entry.")
                total += item.file_size
                if total > 16 * 1024**3:
                    raise ValueError("Checkpoint archive exceeds 16 GiB.")
            zipped.extractall(root)
        candidates = list(root.rglob("modules.json"))
        if len(candidates) != 1:
            raise ValueError("Use the checkpoint ZIP, not the full experiment/checkpoints ZIP.")
        return candidates[0].parent


    def load_e5_checkpoint(path, device="cuda", max_seq_length=512):
        from sentence_transformers import SentenceTransformer

        root = Path(path)
        manifest = json.loads((root / "research_model.json").read_text(encoding="utf-8"))
        if manifest.get("model_family") != "e5" or not manifest.get("fine_tuned"):
            raise ValueError("Select an E5 checkpoint with actual optimizer steps (fine_tuned=true).")
        model = SentenceTransformer(
            str(root), device=device, local_files_only=True, trust_remote_code=False
        )
        model.max_seq_length = max_seq_length
        return model, checkpoint_identity(root, max_seq_length)


    def load_qwen_checkpoint(path, device="cuda", max_seq_length=512):
        """Load a local Qwen3 embedding snapshot; never substitute a base model silently."""
        import torch
        from sentence_transformers import SentenceTransformer

        root = Path(path)
        if not root.is_dir():
            raise ValueError("Select a local Qwen3-Embedding checkpoint directory.")
        manifest_path = root / "research_model.json"
        if (
            manifest_path.is_file()
            and json.loads(manifest_path.read_text("utf-8")).get("format") == "swift_qwen3_adapter_v1"
        ):
            loader = globals().get("load_swift_qwen_adapter")
            if loader is None:
                from notebooks.qwen_adapter_encoder import load_swift_qwen_adapter

                loader = load_swift_qwen_adapter
            return loader(root, max_seq_length)
        config_files = list(root.glob("config.json")) or list(root.glob("0_Transformer/config.json"))
        if (
            not config_files
            or json.loads(config_files[0].read_text(encoding="utf-8")).get("model_type") != "qwen3"
        ):
            raise ValueError("Checkpoint is not a Qwen3 embedding model.")
        model = SentenceTransformer(
            str(root),
            device=device,
            local_files_only=True,
            trust_remote_code=False,
            model_kwargs={"torch_dtype": torch.float16},
        )
        model.max_seq_length = max_seq_length
        if not hasattr(model, "encode_query") or not hasattr(model, "encode_document"):
            raise ValueError("Checkpoint does not support query/document embedding.")
        return model, checkpoint_identity(root, max_seq_length, family="qwen3")

In [ ]:
def stop_embedding_worker():
    import gc

    tunnel = globals().pop("GPU_TUNNEL", None)
    try:
        if tunnel is not None:
            stop_process(tunnel)
    finally:
        server = globals().get("GPU_SERVER")
        if server is not None:
            server.should_exit = True
            GPU_THREAD.join(timeout=60)
            if GPU_THREAD.is_alive():
                raise RuntimeError("GPU đang hoàn tất yêu cầu. Thử lại cell dừng sau.")
            globals().pop("GPU_THREAD", None)
            globals().pop("GPU_SERVER", None)
            del server
            gc.collect()
            import torch

            torch.cuda.empty_cache()
        for name in ("GPU_API_KEY", "GPU_INFO", "GPU_PUBLIC_URL"):
            globals().pop(name, None)

In [ ]:
def start_embedding_worker():
    import tempfile

    import httpx
    import torch
    import uvicorn
    from huggingface_hub import snapshot_download
    from kaggle_secrets import UserSecretsClient

    global GPU_SERVER, GPU_THREAD, GPU_API_KEY, GPU_INFO

    if "GPU_SERVER" in globals():
        raise RuntimeError("Hãy dừng worker hiện tại trước khi chạy lại cell này.")
    if MODEL_FAMILY not in {"qwen3", "e5"} or not 64 <= MAX_SEQ_LENGTH <= 8192:
        raise ValueError("Kiểm tra MODEL_FAMILY và MAX_SEQ_LENGTH.")
    if CHECKPOINT_DIR and CHECKPOINT_ZIP:
        raise ValueError("Chỉ đặt một nguồn checkpoint.")
    if not torch.cuda.is_available():
        raise RuntimeError("Bật GPU trong cài đặt Kaggle trước.")
    try:
        key = UserSecretsClient().get_secret(API_KEY_SECRET_NAME)
    except Exception:
        raise RuntimeError(f"Thêm và cấp quyền cho Kaggle Secret {API_KEY_SECRET_NAME}.") from None
    if not isinstance(key, str) or len(key) < 32 or not all("!" <= c <= "~" for c in key):
        raise ValueError("Dùng khóa ASCII từ 32 ký tự trở lên, ví dụ chuỗi hex ngẫu nhiên.")

    WORKING_DIR.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=WORKING_DIR) as extracted:
        if CHECKPOINT_ZIP:
            unpack = extract_adapter_archive if MODEL_FAMILY == "qwen3" else extract_checkpoint
            selected = unpack(CHECKPOINT_ZIP, extracted)
        elif CHECKPOINT_DIR:
            selected = Path(CHECKPOINT_DIR)
        elif MODEL_FAMILY == "qwen3":
            if BASE_MODEL not in {"Qwen/Qwen3-Embedding-0.6B", "Qwen/Qwen3-Embedding-4B"}:
                raise ValueError("Chọn base Qwen3 Embedding 0.6B hoặc 4B.")
            selected = Path(snapshot_download(BASE_MODEL, revision=BASE_REVISION or None,
                                              ignore_patterns=["*.bin", "*.onnx", "onnx/*"]))
        else:
            raise ValueError("Chọn thư mục hoặc ZIP checkpoint E5 đã huấn luyện.")

        seq_length = MAX_SEQ_LENGTH
        tokenizer_model = E5_TOKENIZER_MODEL if MODEL_FAMILY == "e5" else BASE_MODEL
        if MODEL_FAMILY == "qwen3":
            manifest = selected / "research_model.json"
            if (manifest.is_file()
                    and json.loads(manifest.read_text(encoding="utf-8")).get("format")
                    == "swift_qwen3_adapter_v1"):
                adapter_info = read_adapter_manifest(selected)
                tokenizer_model = adapter_info["base_model"]
                seq_length = adapter_info["max_seq_length"]
            encoder, model_id = load_qwen_checkpoint(selected, "cuda", seq_length)
        else:
            encoder, model_id = load_e5_checkpoint(selected, "cuda", seq_length)
        app = create_embedding_app(
            encoder, key, model_id, seq_length, family=MODEL_FAMILY,
            tokenizer_model=tokenizer_model,
        )

    GPU_SERVER = uvicorn.Server(uvicorn.Config(
        app, host="127.0.0.1", port=PORT, log_level="warning", access_log=False,
    ))
    GPU_THREAD = threading.Thread(target=GPU_SERVER.run, daemon=True)
    try:
        GPU_THREAD.start()
        deadline = time.monotonic() + 60
        while not GPU_SERVER.started and GPU_THREAD.is_alive() and time.monotonic() < deadline:
            time.sleep(0.2)
        if not GPU_SERVER.started:
            raise RuntimeError("Worker chưa khởi động. Kiểm tra cổng và log notebook.")
        headers = {"Authorization": "Bearer " + key}
        with httpx.Client(base_url=f"http://127.0.0.1:{PORT}", trust_env=False) as client:
            client.get("/health", timeout=5).raise_for_status()
            if client.get("/v1/info", timeout=5).status_code != 401:
                raise RuntimeError("Worker chưa yêu cầu xác thực.")
            info = client.get("/v1/info", headers=headers, timeout=5).raise_for_status().json()
            for kind in ("query", "passage"):
                result = client.post("/v1/embeddings", headers=headers, timeout=120, json={
                    "model": model_id, "input": ["Research evidence"], "input_type": kind,
                }).raise_for_status().json()
                if len(result["data"][0]["embedding"]) != info["dimension"]:
                    raise RuntimeError("Kích thước vector không khớp thông tin checkpoint.")
        GPU_API_KEY, GPU_INFO = key, info
        print("GPU:", torch.cuda.get_device_name(0))
        print("Model ID:", info["model"])
        print("Số chiều:", info["dimension"])
        print("Đã kiểm tra API worker bằng yêu cầu query và passage.")
    except BaseException:
        if GPU_THREAD.ident is not None:
            stop_embedding_worker()
        else:
            del GPU_SERVER, GPU_THREAD
        raise


if START_GPU_SERVER:
    start_embedding_worker()
else:
    print("Đặt START_GPU_SERVER = True trong cell cấu hình để chạy worker.")

In [ ]:
if START_CLOUDFLARE:
    import httpx

    if not globals().get("GPU_INFO") or not GPU_SERVER.started or not GPU_THREAD.is_alive():
        raise RuntimeError("Khởi động worker và kiểm tra API nội bộ trước.")
    if "GPU_TUNNEL" in globals() and GPU_TUNNEL.poll() is None:
        raise RuntimeError("Dừng tunnel hiện tại trước khi chạy lại cell này.")
    GPU_TUNNEL, url = start_quick_tunnel(PORT, WORKING_DIR)
    try:
        deadline = time.monotonic() + 45
        with httpx.Client(base_url=url, timeout=8) as client:
            while True:
                try:
                    health = client.get("/health")
                    if health.status_code == 200 and health.json() == {"status": "ok"}:
                        break
                except (httpx.RequestError, ValueError):
                    pass
                if GPU_TUNNEL.poll() is not None or time.monotonic() >= deadline:
                    raise RuntimeError("Tunnel chưa kết nối được với worker Kaggle.")
                time.sleep(1)
            if client.get("/v1/info").status_code != 401:
                raise RuntimeError("Tunnel không trỏ tới worker có xác thực.")
            remote_info = client.get("/v1/info", headers={
                "Authorization": "Bearer " + GPU_API_KEY,
            }).raise_for_status().json()
            if remote_info != GPU_INFO:
                raise RuntimeError("Tunnel trả lại thông tin checkpoint không khớp.")
    except BaseException:
        stop_process(GPU_TUNNEL)
        del GPU_TUNNEL
        raise
    GPU_PUBLIC_URL = url
    print("URL GPU embedding:", GPU_PUBLIC_URL)
    print("Họ model:", GPU_INFO["family"], "· Tokenizer:", GPU_INFO["tokenizer_model"])
    print("Trong Research Assist, mở GPU embedding, chọn model/tokenizer hiển thị ở trên,")
    print("nhập URL và khóa trong Secret, kiểm tra rồi lưu. Tạo lại chỉ mục khi đổi checkpoint.")
else:
    print("Đặt START_CLOUDFLARE = True nếu cần kết nối từ máy khác.")

In [ ]:
STOP_ENDPOINT = False
if STOP_ENDPOINT:
    stop_embedding_worker()
    print("Đã dừng worker GPU và tunnel.")